# Finality test — tuned/engineered baselines on the canonical folds (reproduce §14.13)

**What this reproduces:** the last standing baseline threat, measured on the exact canonical
folds — five tuned/feature-engineered classical methods (`lr_tuned`, `glm_eng`, `lgbm_tuned`,
`cat_tuned`, `rf_tuned`, all tuned by ROC AUC) joining the 9 canonical methods.
Master-report §14.13 verdict: TabPFN stays **#1 of 14 on AUC and PR-AUC on all 6 datasets**;
two small ~1e-4 calibration exceptions.

**How it works:** calls `scripts/eval/insurance_benchmark_v1/run_tuned_baselines.py` — the exact
script behind the addendum. Optionally `analyze_tuned_baselines.py` for the paired stats.

**Requirements:** benchmark-venv kernel + one-click **browser login** in the preflight cell (token saved to the gitignored repo-root `.env`), or `TABPFN_API_KEY` set manually — see `notebooks/reproducibility/README.md`.

**Cost — read this first:** this is the *expensive* notebook. The full 6-dataset suite took hours
(cat_tuned has a 10-min/fold soft cap; early stopping on 5 folds × 6 datasets).
**This notebook runs ONE dataset** (`coil2000`, the smallest) and even that is the slowest cell in
this suite — expect tens of minutes. Run the full suite overnight, one dataset at a time.

**⚠ Warning — append mode:** `run_tuned_baselines.py` **appends** to
`frontier_tuned_baseline_results.csv`. Re-running adds duplicate rows (the analysis layer
de-duplicates by method/dataset; the committed canonical file is the full 14-method suite).
Move the canonical CSV aside before re-running if you want a clean local copy.

## Shipped with executed evidence

This notebook ships with its evidence cells already executed — the rows below are the committed canonical results. Read the outputs to follow along, or re-run any cell: everything except the experiment cell works instantly from the committed `frontier_tuned_baseline_results.csv`.

**The one optional cell is the 'run the experiment' cell (tagged `optional`):** it re-runs the canonical script against the hosted API — tens of minutes for `coil2000` alone, costs credits, and **appends duplicate rows** to the committed CSV. Every other cell is safe and cheap to re-run.

In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *[Path.cwd().parents[i] for i in range(1, 4)]]
            if (p / "scripts/eval/insurance_benchmark_v1/run_tuned_baselines.py").exists())
print("repo root:", ROOT)

repo root: /Users/Scott/Documents/Data Science/ADSWP/TabPFN-work-scott-worktrees/cumans


In [2]:
# Preflight: versions + auth. No TABPFN_API_KEY configured? One-click browser login:
# opens the Prior Labs login page, then saves the token to the repo-root .env
# (gitignored) and exports it for this session's subprocesses.
import importlib, os
for m in ("tabpfn_client", "pandas", "sklearn", "xgboost", "lightgbm", "catboost"):
    mod = importlib.import_module(m)
    print(f"{m:14s} {getattr(mod, '__version__', '?')}")

def _have_key() -> bool:
    if os.environ.get("TABPFN_API_KEY"):
        return True
    env = ROOT / ".env"
    return env.exists() and any(l.startswith("TABPFN_API_KEY=") for l in env.read_text().splitlines())

if not _have_key():
    from tabpfn_client.browser_auth import BrowserAuthHandler
    ok, token = BrowserAuthHandler().try_browser_login()
    assert ok and token, "Browser login failed — see notebooks/reproducibility/README.md"
    env = ROOT / ".env"
    lines = [l for l in env.read_text().splitlines() if not l.startswith("TABPFN_API_KEY=")] if env.exists() else []
    lines.append(f"TABPFN_API_KEY={token}")
    env.write_text("\n".join(lines) + "\n")
    os.environ["TABPFN_API_KEY"] = token
    print("Browser login OK — token saved to repo-root .env (gitignored).")
print("API key present:", _have_key())
assert _have_key(), "no API key — rerun this cell to trigger browser login, or set TABPFN_API_KEY manually"

tabpfn_client  ?
pandas         2.3.3
sklearn        1.6.1
xgboost        3.4.0
lightgbm       4.7.0


catboost       1.2.10
API key present: True


In [ ]:
# OPTIONAL — re-run the experiment (hosted API: tens of minutes, costs credits, appends duplicate rows).
# The exact command behind §14.13, limited to ONE dataset. Drop the dataset argument for the full suite.
import subprocess, time

cmd = [sys.executable, "scripts/eval/insurance_benchmark_v1/run_tuned_baselines.py", "coil2000"]
print("running:", " ".join(cmd), "\n(this is the slow cell — coil2000 alone takes tens of minutes)")
t0 = time.time()
r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-4000:])
print(f"exit: {r.returncode}  ({time.time() - t0:.0f}s)")
if r.returncode:
    print(r.stderr[-2000:])

In [3]:
# The rows appended by the run above (plus any earlier runs).
import pandas as pd
OUT = ROOT / "scripts/eval/insurance_benchmark_v1/frontier_tuned_baseline_results.csv"
df = pd.read_csv(OUT)
print("rows:", len(df), "| methods:", sorted(df.method.unique()))
df.tail(8).round(4)

rows: 180 | methods: ['cat_tuned', 'glm_eng', 'lgbm_tuned', 'lr', 'lr_tuned', 'rf_tuned']


,dataset,method,fold,log_loss,auc,brier,pr_auc,lift10,fit_s,infer_s,config
172,bemtl16,cat_tuned,3,0.2773,0.9496,0.0866,0.8875,2.5795,107.3,0.0,"lr=0.1,depth=10,iters=999"
173,bemtl16,rf_tuned,3,0.2469,0.9541,0.0788,0.8977,2.6174,29.6,0.1,"mf=sqrt,msl=1"
174,bemtl16,lr,4,0.2669,0.9488,0.0859,0.8893,2.5795,73.9,0.0,C=1
175,bemtl16,lr_tuned,4,0.2647,0.9501,0.0850,0.8919,2.5890,6.1,0.0,C=10.0
176,bemtl16,glm_eng,4,0.2603,0.9505,0.0836,0.8933,2.5724,6.7,0.0,"poly=poly2_full(91),C=0.01"
177,bemtl16,lgbm_tuned,4,0.2907,0.9509,0.0863,0.8900,2.5866,1010.9,0.4,"lr=0.05,leaves=127,iters=946 [cap-hit]"
178,bemtl16,cat_tuned,4,0.2811,0.9492,0.0874,0.8870,2.5914,114.8,0.0,"lr=0.1,depth=10,iters=999"
179,bemtl16,rf_tuned,4,0.2544,0.9519,0.0801,0.8922,2.5795,28.7,0.1,"mf=sqrt,msl=1"


## Reading the verdict (§14.13)

- TabPFN is **AUC #1 of 14** on every classification dataset; the closest competitors were
  `glm_eng` (ausprivauto0405, +0.0036, p=0.003), `lgbm_tuned` (bemtl16, +0.0008, p=0.002),
  `cat_tuned` (uslapseagent, +0.0027, p=0.022).
- Two small calibration exceptions (~1e-4, e.g. ausprivauto0405 log loss vs the linear family).
- Honesty note from the report: the tuned GBDTs *regressed vs their own shipped defaults* on
  AUC on 5/6 datasets (norauto: lgbm_tuned 0.640 vs lgbm 0.697) — the credible engineered
  baseline was `glm_eng`, and it still lost every ranking metric.
- TabFM is closed by assessment (OOM at 8GB, non-commercial weights), not by measurement.

Master report §14.13; learning path S9 item 7 / Stage 4.5; digest `docs/MASTER-REPORT-DIGEST.md`.